Techniques that can adapt the number of parameters to represent the pos-
terior online are called adaptive.


Particle ﬁlters are often implemented as a resource-
adaptive algorithm, by adapting the number of particles online based on the
available computational resources.


When applied to ﬁnite spaces, such ﬁlters are known as discrete Bayes
ﬁlters; when applied to continuous spaces, they are commonly called his-
togram ﬁlters.

In [1]:
import numpy as np

bel = np.array([0.25, 0.25, 0.25, 0.25])

motion = np.array([
    [0.7, 0.2, 0.1, 0.0],
    [0.2, 0.6, 0.2, 0.0],
    [0.1, 0.2, 0.6, 0.1],
    [0.0, 0.1, 0.3, 0.6]
])

measurement = np.array([0.9, 0.2, 0.3, 0.5])

# Prediction
bel = motion.T @ bel

# Update
bel = bel * measurement
bel = bel / np.sum(bel)

print(bel)

[0.49180328 0.12021858 0.19672131 0.19125683]


In [2]:
def effective_sample_size(weights):
    return 1.0 / np.sum(weights**2)

In [3]:
def multinomial_resample(particles, weights):
    N = len(particles)
    indices = np.random.choice(range(N), N, p=weights)
    particles = particles[indices]
    weights = np.ones(N) / N
    return particles, weights

In [4]:
def stratified_resample(particles, weights):
    N = len(particles)
    positions = (np.random.rand(N) + range(N)) / N
    indexes = np.zeros(N, 'i')
    
    cumulative_sum = np.cumsum(weights)
    i, j = 0, 0
    
    while i < N:
        if positions[i] < cumulative_sum[j]:
            indexes[i] = j
            i += 1
        else:
            j += 1

    particles = particles[indexes]
    weights = np.ones(N) / N
    return particles, weights

In [5]:
def residual_resample(particles, weights):
    N = len(particles)
    indexes = []

    num_copies = (N * weights).astype(int)
    
    for i in range(N):
        for _ in range(num_copies[i]):
            indexes.append(i)

    residual = weights - num_copies
    residual /= np.sum(residual)

    remaining = N - len(indexes)
    indexes += list(np.random.choice(range(N), remaining, p=residual))

    particles = particles[indexes]
    weights = np.ones(N) / N
    return particles, weights

In [6]:
import numpy as np

N = 200
particles = np.random.uniform(0, 20, N)
weights = np.ones(N) / N

def predict(particles):
    return particles + np.random.normal(0, 1, len(particles))

def update(particles, weights, z):
    weights *= np.exp(-(particles - z)**2)
    weights += 1e-300
    weights /= np.sum(weights)
    return weights

def neff(weights):
    return 1.0 / np.sum(weights**2)

def resample(particles, weights):
    indices = np.random.choice(range(len(particles)), len(particles), p=weights)
    particles = particles[indices]
    weights = np.ones(len(weights)) / len(weights)
    return particles, weights

measurements = [3, 6, 9, 12]

for z in measurements:
    particles = predict(particles)
    weights = update(particles, weights, z)

    if neff(weights) < N/2:
        particles, weights = resample(particles, weights)

    print("Estimate:", np.mean(particles))

Estimate: 2.9868938945985364
Estimate: 5.291603716183577
Estimate: 7.858156248101749
Estimate: 10.494024404188806


In [7]:
import numpy as np

# grid belief
bel = np.array([0.2, 0.3, 0.3, 0.2])

# motion model (shift right)
def motion_update(bel):
    bel_new = np.zeros(len(bel))
    for i in range(len(bel)):
        bel_new[i] = 0.8 * bel[i-1] + 0.2 * bel[i]
    return bel_new

# measurement model
def measurement_update(bel, measurement):
    likelihood = np.array(measurement)
    bel = bel * likelihood
    bel = bel / np.sum(bel)
    return bel

measurements = [
    [0.9, 0.2, 0.1, 0.5],
    [0.2, 0.8, 0.3, 0.4]
]

for m in measurements:
    bel = motion_update(bel)
    bel = measurement_update(bel, m)
    print("Belief:", bel)

Belief: [0.45685279 0.11167513 0.07614213 0.35532995]
Belief: [0.16       0.66075676 0.06681081 0.11243243]


In [8]:
import numpy as np

# initialize particles
N = 100
particles = np.random.uniform(0, 10, N)
weights = np.ones(N) / N

def predict(particles):
    particles += np.random.normal(0, 0.5, len(particles))
    return particles

def update(particles, weights, measurement):
    weights *= np.exp(-(particles - measurement)**2)
    weights += 1e-300
    weights /= np.sum(weights)
    return weights

def resample(particles, weights):
    indices = np.random.choice(range(len(particles)), len(particles), p=weights)
    particles = particles[indices]
    weights = np.ones(len(weights)) / len(weights)
    return particles, weights

measurements = [2, 4, 6, 8]

for z in measurements:
    particles = predict(particles)
    weights = update(particles, weights, z)
    particles, weights = resample(particles, weights)
    print("Estimate:", np.mean(particles))

Estimate: 1.9819887578190136
Estimate: 3.2135479029060314
Estimate: 4.496205379277961
Estimate: 5.190011472952639


In [9]:
def systematic_resample(particles, weights):
    N = len(particles)
    positions = (np.arange(N) + np.random.rand()) / N
    indexes = np.zeros(N, 'i')
    
    cumulative_sum = np.cumsum(weights)
    i, j = 0, 0
    while i < N:
        if positions[i] < cumulative_sum[j]:
            indexes[i] = j
            i += 1
        else:
            j += 1
    
    particles = particles[indexes]
    weights = np.ones(N) / N
    return particles, weights

In [11]:
particles = np.random.uniform(0, 20, 200)
weights = np.ones(200) / 200

for z in [5, 10, 15]:
    # motion
    particles += np.random.normal(0, 1, len(particles))

    # measurement update
    weights *= np.exp(-(particles - z)**2)
    weights /= np.sum(weights)

    # resample
    indices = np.random.choice(range(len(particles)), len(particles), p=weights)
    particles = particles[indices]
    weights = np.ones(len(weights)) / len(weights)

    print("Robot position:", np.mean(particles))

Robot position: 4.869727439745917
Robot position: 7.781627204309673
Robot position: 10.815888622777077
